# 03 — Integration (Harmony)

Integrates Bhaduri 2020 (organoids) and Zhong 2018 (fetal PFC) into a shared embedding using Harmony batch correction.

**Pre-integration cleanup:**
- Remove Bhaduri clusters 9, 10 — choroid plexus (off-target organoid differentiation)
- Remove Bhaduri cluster 16 — unclear / stressed-dying cells
- Remove Zhong cluster 10 — RBC contamination
- **Stress clusters 6 and 13 are kept** — they are a key biological signal (Bhaduri 2020's main finding)

**Integration strategy:**
- Find shared genes between datasets (Bhaduri: 16,774; Zhong: 20,128 post-QC)
- Concatenate on gene intersection
- Joint HVG selection on combined object (top 2,000, `batch_key='dataset'`)
- PCA (30 PCs) → Harmony → UMAP

| Dataset | Cells (post-clustering) | Genes |
|---------|-------------------------|-------|
| Bhaduri et al. 2020 | ~241,777 | 16,774 |
| Zhong et al. 2018 | ~2,330 | 20,128 |

**Prerequisites:** `colab_02_umap_clustering.ipynb` must have been run. Clustered h5ad files must be on Drive.

**Runtime:** Use **High-RAM** (51 GB) — Bhaduri dataset requires it for scaling step.

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install scanpy, leidenalg, and harmonypy
!pip install -q scanpy leidenalg harmonypy

In [ ]:
# Import libraries and define all input/output paths
import os
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white')

DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'

PATHS = {
    'bhaduri_clustered': os.path.join(DRIVE_ROOT, 'data/processed/bhaduri_2020/bhaduri_2020_clustered.h5ad'),
    'zhong_clustered':   os.path.join(DRIVE_ROOT, 'data/processed/zhong_2018/zhong_2018_clustered.h5ad'),
    'integrated':        os.path.join(DRIVE_ROOT, 'data/processed/integrated/integrated_harmony.h5ad'),
}

os.makedirs(os.path.dirname(PATHS['integrated']), exist_ok=True)

print('Paths configured.')
for k, v in PATHS.items():
    print(f'  {k}: {v}')

## 2. Load Clustered Data

In [ ]:
# Load both clustered h5ad files from Drive
adata_organoid = sc.read_h5ad(PATHS['bhaduri_clustered'])
adata_fetal    = sc.read_h5ad(PATHS['zhong_clustered'])

print(f'Bhaduri 2020: {adata_organoid.shape[0]:,} cells x {adata_organoid.shape[1]:,} genes')
print(f'Zhong 2018:   {adata_fetal.shape[0]:,} cells x {adata_fetal.shape[1]:,} genes')
print()
print('Bhaduri obs columns:', list(adata_organoid.obs.columns))
print('Zhong obs columns:  ', list(adata_fetal.obs.columns))

## 2b. Free RAM — Replace Normalized X with Raw Counts

The clustered h5ads store log-normalized X (from the marker gene detection step in colab_02). We need raw counts in X to renormalize the combined object from scratch. `adata.raw` was frozen in colab_01 before any normalization and is preserved through colab_02.

In [ ]:
# Verify adata.raw is present, then replace X with raw counts
if adata_organoid.raw is None:
    raise ValueError('Bhaduri adata.raw is missing — re-run colab_01 and colab_02 to regenerate')
if adata_fetal.raw is None:
    raise ValueError('Zhong adata.raw is missing — re-run colab_01 and colab_02 to regenerate')

adata_organoid.X = adata_organoid.raw.X.copy()
adata_fetal.X    = adata_fetal.raw.X.copy()

gc.collect()

print('X replaced with sparse raw counts for both datasets.')
print(f'Bhaduri X type: {type(adata_organoid.X).__name__}')
print(f'Zhong X type:   {type(adata_fetal.X).__name__}')

## 2c. Fix Zhong X Matrix Type

`adata_fetal.raw.X` was stored as a dense ndarray instead of a sparse matrix.
Converting to `csr_matrix` before concatenation — otherwise `ad.concat` will
densify the entire combined object (~15 GB), crashing the runtime.

In [ ]:
import scipy.sparse as sp
if not sp.issparse(adata_fetal.X):
    adata_fetal.X = sp.csr_matrix(adata_fetal.X)
    print(f'Zhong X converted to {type(adata_fetal.X).__name__}')
else:
    print(f'Zhong X already sparse: {type(adata_fetal.X).__name__}')

## 3. Add Cell Type Annotations

Maps each dataset's Leiden cluster IDs to cell type labels based on the marker gene analysis in colab_02. These labels will be carried into the integrated object so we can color the UMAP by cell type to assess integration quality.

Off-target cell types are labeled here but removed in the next step.

In [ ]:
# Bhaduri cluster annotations (18 clusters, resolution=0.5, from colab_02)
BHADURI_CELL_TYPES = {
    '0':  'vRG',
    '1':  'Mature neurons',
    '2':  'Excitatory neurons',
    '3':  'oRG',
    '4':  'Cycling progenitors',
    '5':  'Cortical progenitors',
    '6':  'Stressed RG',           # kept — key organoid biology
    '7':  'Cycling cortical progenitors',
    '8':  'Excitatory neurons (early)',
    '9':  'Choroid plexus',        # will be removed
    '10': 'Choroid plexus (mixed)', # will be removed
    '11': 'Immature neurons',
    '12': 'GABAergic interneurons',
    '13': 'Stressed progenitors',  # kept — key organoid biology
    '14': 'Astrocyte progenitors',
    '15': 'Excitatory neurons (mature)',
    '16': 'Unclear',               # will be removed
    '17': 'oRG',
}

adata_organoid.obs['cell_type'] = adata_organoid.obs['leiden'].map(BHADURI_CELL_TYPES)

print('Bhaduri cell type distribution:')
print(adata_organoid.obs['cell_type'].value_counts())

In [ ]:
# Zhong cluster annotations (11 clusters, resolution=0.5, from colab_02)
ZHONG_CELL_TYPES = {
    '0':  'Immature neurons',
    '1':  'GABAergic interneurons (MGE)',
    '2':  'Excitatory neurons',
    '3':  'Excitatory neurons (SATB2+)',
    '4':  'Cycling progenitors',
    '5':  'Excitatory neurons',
    '6':  'oRG',
    '7':  'GABAergic interneurons (CGE)',
    '8':  'Microglia',
    '9':  'OPCs',
    '10': 'RBC contamination',     # will be removed
}

adata_fetal.obs['cell_type'] = adata_fetal.obs['leiden'].map(ZHONG_CELL_TYPES)

print('Zhong cell type distribution:')
print(adata_fetal.obs['cell_type'].value_counts())

## 4. Remove Off-Target Clusters

Removing cells that would confound integration:
- **Bhaduri clusters 9, 10** — choroid plexus: off-target organoid differentiation, not a cortical cell type
- **Bhaduri cluster 16** — unclear: likely stressed/dying cells, no informative biology
- **Zhong cluster 10** — RBC contamination: non-neural, will distort the integrated embedding

Stress clusters 6 (stressed RG) and 13 (stressed progenitors) are **kept** — they represent the key organoid-vs-fetal divergence signal we are trying to characterize.

In [ ]:
# Remove off-target clusters from Bhaduri
BHADURI_REMOVE = ['9', '10', '16']
bhaduri_keep = ~adata_organoid.obs['leiden'].isin(BHADURI_REMOVE)

print(f'Bhaduri: removing {(~bhaduri_keep).sum():,} off-target cells '
      f'({(~bhaduri_keep).mean()*100:.1f}%)')
for c in BHADURI_REMOVE:
    n = (adata_organoid.obs['leiden'] == c).sum()
    label = BHADURI_CELL_TYPES[c]
    print(f'  cluster {c} ({label}): {n:,} cells')

print()

# Remove RBC cluster from Zhong
ZHONG_REMOVE = ['10']
zhong_keep = ~adata_fetal.obs['leiden'].isin(ZHONG_REMOVE)

print(f'Zhong: removing {(~zhong_keep).sum():,} RBC cells '
      f'({(~zhong_keep).mean()*100:.1f}%)')
for c in ZHONG_REMOVE:
    n = (adata_fetal.obs['leiden'] == c).sum()
    label = ZHONG_CELL_TYPES[c]
    print(f'  cluster {c} ({label}): {n:,} cells')

adata_organoid = adata_organoid[bhaduri_keep].copy()
adata_fetal    = adata_fetal[zhong_keep].copy()

print()
print(f'After removal — Bhaduri: {adata_organoid.shape[0]:,} cells')
print(f'After removal — Zhong:   {adata_fetal.shape[0]:,} cells')

## 5. Gene Intersection

Bhaduri has 16,774 genes and Zhong has 20,128 genes post-QC. Integration requires a common gene space. We take the intersection of shared genes — genes present in both datasets.

In [ ]:
# Find shared genes and subset both datasets to the intersection
shared_genes = np.intersect1d(adata_organoid.var_names, adata_fetal.var_names)

print(f'Bhaduri genes:          {adata_organoid.shape[1]:,}')
print(f'Zhong genes:            {adata_fetal.shape[1]:,}')
print(f'Shared genes:           {len(shared_genes):,}')
print(f'Lost from Bhaduri:      {adata_organoid.shape[1] - len(shared_genes):,} (not in Zhong)')
print(f'Lost from Zhong:        {adata_fetal.shape[1] - len(shared_genes):,} (not in Bhaduri)')

adata_organoid = adata_organoid[:, shared_genes].copy()
adata_fetal    = adata_fetal[:, shared_genes].copy()

print()
print(f'After subsetting — Bhaduri: {adata_organoid.shape}')
print(f'After subsetting — Zhong:   {adata_fetal.shape}')

## 6. Concatenate

Merge both objects into a single AnnData. The `label='dataset'` argument adds an `obs` column `'dataset'` with values `'bhaduri'` or `'zhong'` — this is the batch key for Harmony.

In [ ]:
# Prepend dataset name to barcodes to prevent obs_name collisions
adata_organoid.obs_names = 'bhaduri_' + adata_organoid.obs_names
adata_fetal.obs_names    = 'zhong_'   + adata_fetal.obs_names

print('First 3 Bhaduri barcodes:', adata_organoid.obs_names[:3].tolist())
print('First 3 Zhong barcodes:  ', adata_fetal.obs_names[:3].tolist())

In [ ]:
# Concatenate — anndata.concat creates a combined object with a 'dataset' obs column
adata = ad.concat(
    {'bhaduri': adata_organoid, 'zhong': adata_fetal},
    label='dataset',
    join='inner',    # keep only shared genes (already done above — safety net)
    merge='first'    # for conflicting var metadata, take from first dataset
)

print(f'Combined object: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes')
print()
print('Dataset breakdown:')
print(adata.obs['dataset'].value_counts())
print()
print('obs columns:', list(adata.obs.columns))

## 7. Normalize + HVG Selection on Combined Object

In [ ]:
# Normalize to 10k counts per cell, then log-transform
# Starting from raw counts (X was replaced with adata.raw.X earlier)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print('Combined: log-normalized.')

In [ ]:
# Freeze the log-normalized state as adata.raw — needed for:
#   (a) visualization with use_raw=True (all shared genes available, not just HVGs)
#   (b) downstream RNA velocity (colab_06)
# This MUST happen before scale() and before filtering to HVGs.
adata.raw = adata
print(f'adata.raw set: {adata.raw.shape[0]:,} cells x {adata.raw.shape[1]:,} genes (log-normalized, all shared genes)')

In [ ]:
# Joint HVG selection on the combined object
# batch_key='dataset' selects genes that are variable within each batch independently,
# not just genes that differ because the two datasets have different mean expression.
# This is critical for finding biologically meaningful genes shared across datasets.
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    batch_key='dataset',
    flavor='seurat',    # works with log-normalized X
    subset=False        # mark with highly_variable column, don't subset yet
)

n_hvg = adata.var['highly_variable'].sum()
print(f'HVGs selected: {n_hvg:,} out of {adata.shape[1]:,} shared genes')

# Scatter plot showing which genes were selected
sc.pl.highly_variable_genes(adata)

In [ ]:
# Subset to HVGs and scale
# After subsetting: adata.var_names = 2000 HVGs; adata.raw.var_names = all shared genes
adata = adata[:, adata.var['highly_variable']].copy()
print(f'After HVG subsetting: {adata.shape}')

# Scale to zero mean, unit variance per gene — clip extreme values at 10 standard deviations
# (makes PCA less sensitive to outlier cells)
sc.pp.scale(adata, max_value=10)
print('Scaled.')

## 8. PCA

30 PCs — a conservative choice for a combined two-dataset object. We'll inspect the elbow plot and adjust if clearly under/overshooting.

In [ ]:
# PCA on the combined HVG-scaled matrix
sc.tl.pca(adata, n_comps=30, svd_solver='arpack')
print('PCA done.')

# Elbow plot — expect gradual decline; 30 PCs should capture most variance
sc.pl.pca_variance_ratio(adata, n_pcs=30, log=True)

## 9. Unintegrated UMAP (before Harmony)

Computed on the raw PCA (no batch correction). If the batch effect is large, the two datasets will form separate regions even when colored by cell type.

We store this as `obsm['X_umap_unintegrated']` so we can compare side-by-side with the Harmony UMAP later.

In [ ]:
# Build neighbor graph on the unintegrated PCA
sc.pp.neighbors(adata, use_rep='X_pca', n_neighbors=30, n_pcs=30,
                key_added='neighbors_unintegrated')
print('Unintegrated neighbor graph built.')

In [ ]:
# Compute UMAP on unintegrated graph — store separately from the post-Harmony UMAP
sc.tl.umap(adata, neighbors_key='neighbors_unintegrated')
adata.obsm['X_umap_unintegrated'] = adata.obsm['X_umap'].copy()

print('Unintegrated UMAP stored in obsm["X_umap_unintegrated"].')

# Color by dataset — if datasets separate strongly, batch correction is needed
sc.pl.embedding(adata, basis='umap_unintegrated', color='dataset',
                title='Before Harmony — colored by dataset')

# Color by cell type — do same cell types already cluster together, or are they split by dataset?
sc.pl.embedding(adata, basis='umap_unintegrated', color='cell_type',
                title='Before Harmony — colored by cell type')

## 10. Harmony Integration

Harmony corrects batch effects in PCA space by iteratively adjusting cluster centroids to be dataset-balanced.

- **Input:** `X_pca` (unintegrated)
- **Output:** `X_pca_harmony` (batch-corrected PCA)
- **`theta=2`** (default) — controls diversity penalty strength. If integration looks undercorrected (datasets still fully separate), increase to 3–4. If overcorrected (datasets fully collapsed), decrease to 1.

Harmony does **not** modify the expression matrix — only the PCA embedding used for UMAP and clustering.

In [ ]:
# Run Harmony directly via harmonypy (avoids scanpy wrapper shape bug)
# Z_corr is returned as (n_cells, n_pcs) in current harmonypy — assign directly
import harmonypy as hm

ho = hm.run_harmony(
    adata.obsm['X_pca'],
    adata.obs,
    'dataset',
    theta=2,
    max_iter_harmony=50,    # 50 to ensure convergence (converged at 33 in practice)
)
adata.obsm['X_pca_harmony'] = ho.Z_corr   # shape: (n_cells, n_pcs)

print('Harmony integration complete.')
print(f'Corrected PCA shape: {adata.obsm["X_pca_harmony"].shape}')

## 11. Integrated UMAP

Build the neighbor graph on the Harmony-corrected embedding, then compute UMAP. This overwrites the current `X_umap` (the unintegrated version is safely stored in `X_umap_unintegrated`).

In [ ]:
# Build neighbor graph on the Harmony-corrected PCA
# n_neighbors=30 — same as unintegrated, for fair visual comparison
sc.pp.neighbors(adata, use_rep='X_pca_harmony', n_neighbors=30)
print('Harmony neighbor graph built.')

In [ ]:
# Compute UMAP on the Harmony-corrected graph
# (takes 5–15 min on 242k+ cells)
sc.tl.umap(adata)
print('Integrated UMAP computed (stored in obsm["X_umap"]).')

## 12. Leiden Clustering on Integrated Object

Clusters the cells using the Harmony-corrected neighbor graph. Clusters here reflect shared cell type structure across both datasets, not just within one.

In [ ]:
# Leiden clustering at resolution=0.5 — start conservative, adjust after inspecting UMAP
LEIDEN_RES = 0.5

sc.tl.leiden(adata, resolution=LEIDEN_RES)

n_clusters = adata.obs['leiden'].nunique()
print(f'Integrated Leiden: {n_clusters} clusters at resolution={LEIDEN_RES}')
print(adata.obs['leiden'].value_counts().sort_index())

## 13. Visualize Integration Quality

Three checks:
1. **Before vs after Harmony** colored by `dataset` — are cells from both datasets mixing?
2. **Cell types** on integrated UMAP — do the same cell types cluster together across datasets?
3. **Leiden clusters** — how does the integrated clustering map onto the original annotations?

**What to look for:**
- Good integration: bhaduri and zhong cells intermix within each cell type region
- Over-integration: datasets fully collapsed, no organoid-vs-fetal separation visible anywhere — biological signal erased
- Under-integration: datasets still completely separate even for identical cell types

**If undercorrected:** re-run Harmony with `theta=3` or `theta=4`  
**If overcorrected:** re-run Harmony with `theta=1`, or fall back to scVI

In [ ]:
# Before Harmony
sc.pl.embedding(adata, basis='umap_unintegrated', color='dataset',
                title='Before Harmony — by dataset')

# After Harmony
sc.pl.umap(adata, color='dataset', title='After Harmony — by dataset')

In [ ]:
# Cell type annotations on the integrated UMAP
# Same cell types (e.g. vRG, oRG, excitatory neurons) should co-localize across datasets
sc.pl.umap(adata, color='cell_type', legend_loc='on data',
           legend_fontsize=6, title='Integrated UMAP — cell type')

In [ ]:
# Integrated Leiden clusters
sc.pl.umap(adata, color='leiden', legend_loc='on data',
           title='Integrated UMAP — Leiden clusters')

In [ ]:
# Dataset composition per Leiden cluster — reveals which clusters are dataset-specific
# Stress clusters (Bhaduri-only) should appear as bhaduri-dominant clusters
# Microglia/OPCs (Zhong-only) should appear as zhong-dominant clusters

cluster_composition = (
    adata.obs.groupby(['leiden', 'dataset'])
    .size()
    .unstack(fill_value=0)
)
cluster_composition_pct = cluster_composition.div(cluster_composition.sum(axis=1), axis=0) * 100
cluster_composition_pct = cluster_composition_pct.round(1)

print('Dataset composition per Leiden cluster (%):')    
print(cluster_composition_pct.to_string())

In [ ]:
# Color by gestational week (Zhong cells only) — should see temporal gradient
# NaN for all Bhaduri cells, which scanpy renders as grey
if 'gestational_week' in adata.obs.columns:
    sc.pl.umap(adata, color='gestational_week',
               title='Integrated UMAP — gestational week (Zhong only)')
else:
    print('gestational_week column not found — was it created in colab_02?')

In [ ]:
# Color by Bhaduri sample ID (NaN for Zhong cells)
# Checks whether samples from the same protocol land in similar regions
if 'sample' in adata.obs.columns:
    sc.pl.umap(adata, color='sample',
               title='Integrated UMAP — Bhaduri sample ID')
else:
    print('sample column not found — was it extracted in colab_02?')

## 14. Known Brain Markers on Integrated UMAP

Using `use_raw=True` to pull expression from `adata.raw` — this gives log-normalized values for all shared genes, not just the 2,000 HVGs used for PCA.

In [ ]:
# Known brain cell type markers
brain_markers = [
    'SOX2',    # radial glia / neural progenitors
    'PAX6',    # radial glia
    'EOMES',   # intermediate progenitors (TBR2)
    'TBR1',    # early excitatory neurons
    'NEUROD2', # excitatory neurons
    'GAD1',    # inhibitory neurons
    'GAD2',    # inhibitory neurons
    'GFAP',    # astrocytes
    'MKI67',   # cycling cells
]

# Check which markers are in the shared gene set (adata.raw contains all shared genes)
available = [g for g in brain_markers if g in adata.raw.var_names]
missing   = [g for g in brain_markers if g not in adata.raw.var_names]
print(f'Available markers: {available}')
print(f'Missing from shared genes: {missing}')

if available:
    sc.pl.umap(adata, color=available, use_raw=True, ncols=3,
               title=[f'Integrated — {g}' for g in available])

## 15. Save Integrated Object

In [ ]:
# Save the integrated AnnData to Drive
# Contains: Harmony-corrected PCA (X_pca_harmony), integrated UMAP (X_umap),
# unintegrated UMAP (X_umap_unintegrated), Leiden clusters, cell type annotations,
# and adata.raw with log-normalized counts for all shared genes

adata.write_h5ad(PATHS['integrated'])

size_mb = os.path.getsize(PATHS['integrated']) / 1e6
print(f'Saved: {PATHS["integrated"]} ({size_mb:.1f} MB)')

print()
print('Object summary:')
print(f'  Cells:             {adata.shape[0]:,}')
print(f'  Genes (HVG set):   {adata.shape[1]:,}')
print(f'  Genes (raw):       {adata.raw.shape[1]:,}')
print(f'  Leiden clusters:   {adata.obs["leiden"].nunique()}')
print(f'  obs keys:          {list(adata.obs.columns)}')
print(f'  obsm keys:         {list(adata.obsm.keys())}')

## Done

Both datasets have been integrated:
- Off-target cells removed (choroid plexus, RBC, unclear)
- Subsetted to ~16,000 shared genes
- 2,000 HVGs selected jointly (batch-aware)
- PCA (30 PCs) → Harmony → integrated UMAP + Leiden clusters

Saved to Drive: `data/processed/integrated/integrated_harmony.h5ad`

**Inspect the integration quality:**
- Do the same cell types (vRG, oRG, excitatory neurons) intermix across datasets?
- Are the organoid stress clusters (Stressed RG, Stressed progenitors) present in the integrated space? Where do they land relative to fetal cell types?
- Is there still some dataset separation visible (expected — organoids and fetal brain are genuinely different)?

**If undercorrected** (datasets fully separate): re-run Harmony with `theta=3`  
**If overcorrected** (no biological separation): re-run with `theta=1` or switch to scVI

**Next steps:**
- `colab_04_cell_type_annotation.ipynb` — annotate clusters in the integrated space, reannotate if integration changed cluster composition
- `colab_05_trajectory.ipynb` — Monocle3 pseudotime on the integrated embedding
- `colab_06_rna_velocity.ipynb` — scVelo (requires re-running data loading with spliced/unspliced matrices)